In [1]:
from paddleocr import PaddleOCR
import matplotlib.pyplot as plt
import re
import warnings
warnings.filterwarnings("ignore")

Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


## Initialize the OCR and run it

In [4]:
ocr = PaddleOCR(lang="en")
image = "receipt.png"
result = ocr.ocr(image)
print(result)

Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Shahad\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Shahad\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Shahad\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Shahad\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Shahad\.pad

[{'input_path': 'receipt.png', 'page_index': None, 'doc_preprocessor_res': {'input_path': None, 'page_index': None, 'input_img': array([[[236, ..., 239],
        ...,
        [ 59, ...,  68]],

       ...,

       [[229, ..., 235],
        ...,
        [147, ..., 162]]], dtype=uint8), 'model_settings': {'use_doc_orientation_classify': True, 'use_doc_unwarping': True}, 'angle': 0, 'rot_img': array([[[236, ..., 239],
        ...,
        [ 59, ...,  68]],

       ...,

       [[229, ..., 235],
        ...,
        [147, ..., 162]]], dtype=uint8), 'output_img': array([[[243, ..., 245],
        ...,
        [241, ..., 241]],

       ...,

       [[246, ..., 248],
        ...,
        [234, ..., 236]]], dtype=uint8)}, 'dt_polys': [array([[14, 11],
       ...,
       [13, 44]], dtype=int16), array([[164,  41],
       ...,
       [163,  74]], dtype=int16), array([[149,  67],
       ...,
       [148, 100]], dtype=int16), array([[108,  93],
       ...,
       [107, 126]], dtype=int16), array([[

In [5]:
text_lines =[]
for line in result:
    for word in line["rec_texts"]:
        text_lines.append(word)
if len(text_lines) == 0:
    vendor, date, total = None, None, None

invalid_vendor_words = ["TAX", "INVOICE", "RECEIPT"]

vendor = text_lines[0] if len(text_lines) > 0 else None

if vendor and any(w in vendor.upper() for w in invalid_vendor_words):
    vendor = text_lines[1] if len(text_lines) > 1 else vendor

if vendor:
    vendor = vendor.strip().upper()

date = None

date_patterns = [
    r"\d{2}-\d{2}-\d{2}",
    r"\d{2}/\d{2}/\d{2}",
    r"\d{2}/\d{2}/\d{4}",
    r"\d{4}-\d{2}-\d{2}",
    r"\d{2}\.\d{2}\.\d{2}"
]

prices = []

for line in text_lines:

    # Detect dates
    for pattern in date_patterns:
        match = re.search(pattern, line)
        if match:
            date = match.group()

    # Detect prices like 23.40 or 75,000
    m = re.search(r"\d+[.,]\d{2,3}", line)

    if m:
        value = float(m.group().replace(",", ""))
        prices.append(value)

if prices:
    total = max(prices)
else:
    total = None


print("Vendor: ", vendor)
print("Date: ", date)
print("Total: ", total)

Vendor:  GARDENIA BAKERIES (KLLSDN BHD (139386 X)
Date:  12/09/2017
Total:  105.49
